# n-step returns, the λ-return, and eligibility traces (TD(λ))

_Reinforcement Learning_

**One dial, λ, slides smoothly between one-step TD and full Monte Carlo — and spreads credit back over a whole trajectory.**

One-step TD and Monte Carlo are the two ends of a single idea: how far ahead do you look at real rewards before you fall back on your own value estimate?

       
         
- One step (TD): use one real reward $R_{t+1}$, then trust your guess $V(S_{t+1})$ for everything after. This is "bootstrapping" — building an estimate on top of another estimate.
         
- All the way (MC): use the actual rewards $R_{t+1}, R_{t+2}, \dots$ for the whole rest of the episode, and never lean on a guess.
         
- Any number of steps in between ($n$-step): use $n$ real rewards, then bootstrap from $V(S_{t+n})$.
       

---

This notebook is a practice scaffold for the **n-step returns, the λ-return, and eligibility traces (TD(λ))** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — numpy

We implement tabular **TD(λ)** with accumulating eligibility traces on the classic 5-state random walk, then compare a few values of λ against the known true state values. We build it in three steps: (1) the random-walk environment, (2) the TD(λ) update with eligibility traces, (3) sweeping λ to see its effect.

### Step 1 — Build the 5-state random walk

States $S_1\dots S_5$ sit in a line with terminals on both ends. Every episode starts in the middle ($S_3$) and steps left or right with equal probability. Falling off the **left** end gives reward 0; off the **right** end gives `+1`. Under this random policy the true state values are $1/6, 2/6, \dots, 5/6$, which we store in `TRUE_V` to score our estimates against.

In [ ]:
import numpy as np

# ---- the 5-state random walk: S1..S5, terminals on both ends ----
# Start in the middle (S3). Each step goes left/right with prob 0.5.
# Stepping off the LEFT end gives reward 0; off the RIGHT end gives +1.
# True state values under the random policy: [1/6, 2/6, 3/6, 4/6, 5/6].
N = 5
TRUE_V = np.arange(1, N + 1) / (N + 1)   # [0.167, 0.333, 0.5, 0.667, 0.833]

def run_episode(rng):
    """Return a list of (state, reward, next_state) with next_state=None at a terminal."""
    s = N // 2                           # start in the middle (S3, index 2)
    traj = []
    while True:
        ns = s - 1 if rng.random() < 0.5 else s + 1
        if ns < 0:                       # fell off the left end -> reward 0, terminal
            traj.append((s, 0.0, None))
            break
        if ns >= N:                      # fell off the right end -> reward +1, terminal
            traj.append((s, 1.0, None))
            break
        traj.append((s, 0.0, ns))
        s = ns
    return traj

### Step 2 — The TD(λ) update with eligibility traces

Each step computes the usual one-step TD error $\delta = r + \gamma V(S') - V(S)$. The trick is the **eligibility trace** vector `e`: visiting a state bumps its trace by 1, and *every* state's value is nudged by `alpha * delta * e`. After the update all traces decay by $\gamma\lambda$. So a single TD error reaches back and credits recently visited states, with λ controlling how far back that credit reaches.

In [ ]:
def td_lambda(lam, alpha=0.05, n_episodes=100, gamma=1.0, seed=0):
    """Tabular TD(lambda) prediction with ACCUMULATING eligibility traces."""
    rng = np.random.RandomState(seed)
    V = np.full(N, 0.5)                   # optimistic-ish init at 0.5
    for _ in range(n_episodes):
        e = np.zeros(N)                  # eligibility traces, reset each episode
        for (s, r, ns) in run_episode(rng):
            v_next = 0.0 if ns is None else V[ns]
            delta = r + gamma * v_next - V[s]        # one-step TD error
            e[s] += 1.0                              # accumulate trace for visited state
            V += alpha * delta * e                   # broadcast error scaled by trace
            e *= gamma * lam                         # decay all traces by gamma*lambda
    return V

### Step 3 — Sweep λ and score against the true values

Run TD(λ) for a few λ values and measure the root-mean-square error of the learned values against `TRUE_V`. `λ=0` is plain TD(0) (one-step bootstrapping); `λ=1` is essentially every-visit Monte Carlo. The interesting region is *between* them — an intermediate λ typically needs fewer episodes to reach a given error.

In [ ]:
# Compare a couple of lambda values against the true values.
for lam in [0.0, 0.8, 1.0]:              # 0.0 = TD(0), 1.0 ~ every-visit MC
    V = td_lambda(lam, alpha=0.05, n_episodes=100, seed=0)
    rms = np.sqrt(np.mean((V - TRUE_V) ** 2))
    print(f"lambda={lam:.1f}  V={np.round(V, 3).tolist()}  RMS={rms:.3f}")

print("true V =", np.round(TRUE_V, 3).tolist())
# lambda=0.0  V=[0.12, 0.304, 0.468, 0.617, 0.794]  RMS=0.06x
# lambda=0.8  V=[0.112, 0.289, 0.423, 0.569, 0.789]  RMS=0.07x
# An intermediate lambda needs FEWER episodes to reach a given error (see CODEVIZ).

## Visualize the data & results

_On the 5-state random walk, how does value-estimate error after 10 episodes depend on λ — does an intermediate λ really beat both TD(λ=0) and MC(λ=1)?_

We re-use the same random walk and TD(λ) routine, but now sweep a fine grid of λ values and average the error over many independent runs. We do it in two steps: (1) restate the environment and TD(λ) update, (2) sweep λ and average the RMS error to reveal the U-shape.

### Step 1 — Restate the random walk and TD(λ) update

This is the same environment and the same accumulating-trace update as the reference, repeated here so the visualization cell stands on its own. The episode generator returns `(state, reward, next_state)` triples and `td_lambda` runs the trace-based update over a given number of episodes.

In [ ]:
import numpy as np

# 5-state random walk; true values 1/6..5/6 (Sutton & Barto Example 6.2).
N = 5
TRUE_V = np.arange(1, N + 1) / (N + 1)

def run_episode(rng):
    s = N // 2
    traj = []
    while True:
        ns = s - 1 if rng.random() < 0.5 else s + 1
        if ns < 0:
            traj.append((s, 0.0, None))
            break
        if ns >= N:
            traj.append((s, 1.0, None))
            break
        traj.append((s, 0.0, ns))
        s = ns
    return traj

def td_lambda(lam, alpha, n_episodes, seed, gamma=1.0):
    rng = np.random.RandomState(seed)
    V = np.full(N, 0.5)
    for _ in range(n_episodes):
        e = np.zeros(N)
        for (s, r, ns) in run_episode(rng):
            v_next = 0.0 if ns is None else V[ns]
            delta = r + gamma * v_next - V[s]
            e[s] += 1.0                       # accumulating trace
            V += alpha * delta * e
            e *= gamma * lam
    return V

### Step 2 — Sweep λ and average the error to reveal the U-shape

For each λ we run TD(λ) for just `K=10` episodes, measure the RMS error against the true values, and average over 200 independent runs to smooth out randomness. Plotting RMS against λ gives a **U-shape**: an interior λ around 0.8 beats both λ=0 (pure TD) and λ=1 (pure MC), confirming that blending real rewards with bootstrapping learns fastest here.

In [ ]:
lambdas = [0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]
K = 10            # error measured after K episodes
alpha = 0.05
n_runs = 200      # average over this many independent runs

rms_by_lambda = []
for lam in lambdas:
    errs = [np.sqrt(np.mean((td_lambda(lam, alpha, K, seed=run) - TRUE_V) ** 2))
            for run in range(n_runs)]
    rms_by_lambda.append(round(float(np.mean(errs)), 4))

print("lambdas:", lambdas)
print("rms    :", rms_by_lambda)
# rms: [0.1745, 0.1686, 0.1623, 0.156, 0.1534, 0.1523, 0.156, 0.1635, 0.1831]
# -> U-shape: interior lambda ~0.8 beats both lambda=0 (TD) and lambda=1 (MC).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Write out the 1-step, 2-step, and 3-step returns $G_t^{(1)},G_t^{(2)},G_t^{(3)}$ for a state at time $t$, and say which is closest to a Monte Carlo update.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use the definition $G_t^{(n)} = R_{t+1}+\gamma R_{t+2}+\dots+\gamma^{n-1}R_{t+n}+\gamma^{n}V(S_{t+n})$ for $n=1,2,3$. — _Each adds one more real reward before bootstrapping from $V$._
- Note that larger $n$ uses more actual rewards and relies less on the estimate $V$. — _More real data = closer to Monte Carlo, which uses ALL the rewards._

**Answer:** $G_t^{(1)} = R_{t+1} + \gamma V(S_{t+1})$ (this is the TD(0) target). $G_t^{(2)} = R_{t+1} + \gamma R_{t+2} + \gamma^2 V(S_{t+2})$. $G_t^{(3)} = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \gamma^3 V(S_{t+3})$. The 3-step return is closest to Monte Carlo, because it uses the most real rewards before falling back on the bootstrap estimate $V$.

</details>

**Problem 2.** With $\lambda=0.5$, what fraction of the $\lambda$-return's weight sits on the 1-step return, and what does $\lambda=0$ do to the $\lambda$-return?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The weight on the $n$-step return is $(1-\lambda)\lambda^{n-1}$. Plug in $n=1,\ \lambda=0.5$. — _That gives the share going to the one-step term._
- Set $\lambda=0$ in $(1-\lambda)\lambda^{n-1}$ for $n=1$ vs $n\ge2$. — _$0^0=1$ but $0^{k}=0$ for $k\ge1$, so only the first term survives._

**Answer:** Weight on the 1-step return is $(1-0.5)\cdot0.5^{0} = 0.5$ — half the total. The rest decays geometrically ($0.25$ on the 2-step, $0.125$ on the 3-step, …). Setting $\lambda=0$ puts weight $(1-0)\cdot0^{0}=1$ on the 1-step return and $0$ on all others, so $G_t^{\lambda}=G_t^{(1)}$ — the $\lambda$-return collapses to pure one-step TD.

</details>

**Problem 3.** In a tabular run with $\gamma=1,\lambda=0.8$, you visit $S_a$ then $S_b$ then hit a reward giving TD error $\delta=2$. The traces are $e(S_a)=0.8,\ e(S_b)=1.0$ and $\alpha=0.1$. How much does each value change?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply $V(s)\leftarrow V(s)+\alpha\,\delta\,e(s)$ to each state. — _Eligibility traces broadcast the single TD error to all states in proportion to their trace._
- Compute $\alpha\,\delta\,e$ for $S_a$ and $S_b$. — _$S_b$ (more recent, bigger trace) gets more credit than $S_a$._

**Answer:** $\Delta V(S_b) = 0.1\cdot 2\cdot 1.0 = 0.2$ and $\Delta V(S_a) = 0.1\cdot 2\cdot 0.8 = 0.16$. The one surprise credits both states in a single update — the more recently visited $S_b$ gets more. Plain TD(0) would have updated only the state where the error occurred.

</details>